# Site-fraction composition on multiple sublattices

## Prerequisites

See the [site-fraction mixture lecture note](../../../docs/lecture-notes/materials/mixtures/sublattice-site-fractions/index.md). This laboratory assumes normalized fractions, formula-unit multiplicities, and the distinction between chemical species and an explicitly represented vacancy pseudocomponent.

## Objectives

1. Define two ordered sublattices with independent species sets and multiplicities.
2. Normalize site counts independently on each sublattice.
3. Evaluate multiplicity-weighted overall amounts per formula unit.
4. Exclude the vacancy pseudocomponent explicitly when forming constituent mole fractions.
5. Visualize site fractions separately from overall composition.

All labels and counts are synthetic instructional inputs.


## Definitions

For sublattice $s$ with multiplicity $a_s$, site count $n_i^{(s)}$, and total site count $N_s$, the site fraction is

$$
y_i^{(s)}=\frac{n_i^{(s)}}{N_s},
\qquad
\sum_{i\in\mathcal A_s}y_i^{(s)}=1.
$$

Multiplicity-weighted amounts per formula unit are

$$
N_i^{\mathrm{fu}}=\sum_s a_s y_i^{(s)}.
$$

If a declared set $\mathcal E$ of pseudocomponents is excluded, the included constituent fractions are

$$
X_i=\frac{N_i^{\mathrm{fu}}}{\sum_{j\notin\mathcal E}N_j^{\mathrm{fu}}},
\qquad i\notin\mathcal E.
$$

Each sublattice is normalized independently; the state space is a product of simplices, not one global simplex.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.materials.mixtures.sublattice import (
    SiteCountMixtureNormalizationRequest,
    SiteCountMixtureNormalizer,
    SublatticeSpecification,
)
from projectkoios.physkit.units import ScalarQuantity, Unitless, VectorQuantity

In [ ]:
specifications = (
    SublatticeSpecification(
        name="substitutional",
        species=("A", "B"),
        multiplicity=ScalarQuantity(magnitude=1.0, unit=Unitless()),
    ),
    SublatticeSpecification(
        name="interstitial",
        species=("B", "Va"),
        multiplicity=ScalarQuantity(magnitude=2.0, unit=Unitless()),
    ),
)
normalization = SiteCountMixtureNormalizer().action(
    request=SiteCountMixtureNormalizationRequest(
        specifications=specifications,
        site_counts=(
            VectorQuantity(
                magnitude=np.asarray([3.0, 1.0], dtype=np.float64),
                unit=Unitless(),
            ),
            VectorQuantity(
                magnitude=np.asarray([2.0, 6.0], dtype=np.float64),
                unit=Unitless(),
            ),
        ),
    )
)
mixture = normalization.mixture
overall = mixture.evaluate_overall_composition(excluded_species=("Va",))

In [ ]:
assert np.allclose(
    mixture.sublattices[0].fractions.magnitude,
    np.asarray([0.75, 0.25], dtype=np.float64),
)
assert np.allclose(
    mixture.sublattices[1].fractions.magnitude,
    np.asarray([0.25, 0.75], dtype=np.float64),
)
assert overall.included_species == ("A", "B")
assert np.allclose(
    overall.amounts_per_formula_unit.magnitude,
    np.asarray([0.75, 0.75], dtype=np.float64),
)
assert np.allclose(
    overall.mole_fractions.magnitude,
    np.asarray([0.5, 0.5], dtype=np.float64),
)

In [ ]:
display_species = ("A", "B", "Va")
site_fraction_matrix = np.asarray(
    [
        [
            (
                sublattice.fraction_for(species_name=species_name).magnitude
                if species_name in sublattice.specification.species
                else 0.0
            )
            for species_name in display_species
        ]
        for sublattice in mixture.sublattices
    ],
    dtype=np.float64,
)
figure, (site_axis, overall_axis) = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(11.0, 4.5),
)
bottom = np.zeros(len(specifications), dtype=np.float64)
for species_index, species_name in enumerate(display_species):
    values = site_fraction_matrix[:, species_index]
    site_axis.bar(
        np.arange(len(specifications)),
        values,
        bottom=bottom,
        label=species_name,
    )
    bottom = bottom + values
site_axis.set(
    xticks=np.arange(len(specifications)),
    xticklabels=tuple(specification.name for specification in specifications),
    ylabel="site fraction",
    title="Independent sublattice simplices",
)
site_axis.set_ylim(0.0, 1.0)
site_axis.legend()
site_axis.grid(True, axis="y")

overall_axis.bar(
    overall.included_species,
    overall.mole_fractions.magnitude,
    color=("tab:blue", "tab:orange"),
)
overall_axis.set(
    ylabel="included constituent mole fraction",
    title="Overall composition with Va excluded",
)
overall_axis.set_ylim(0.0, 1.0)
overall_axis.grid(True, axis="y")
figure.tight_layout()
plt.show()

## Interpretation

The second sublattice has twice the formula-unit multiplicity of the first, so its site fractions contribute with twice the weight to the overall amounts. Vacancy exclusion is part of the explicit evaluation request rather than an implicit universal policy.

This laboratory verifies the represented algebra only. It does not identify a real crystal structure, establish equilibrium site fractions, or supply a CALPHAD thermodynamic model.

## Exercises

1. Evaluate the overall composition without excluding `Va` and interpret the resulting normalized pseudocomponent fraction.
2. Change the second multiplicity from two to one and predict the overall constituent fractions before executing.
3. Exchange the two count vectors while preserving their sublattice specifications.
4. Explain what additional thermodynamic model would be required to determine equilibrium site fractions.
